# Contraste de H4: geografía y dependencia en establecimientos rurales y urbanos

> **H4.** En los establecimientos rurales las variables geográficas presentan mayor contribución que la dependencia administrativa, y en los urbanos ocurre lo inverso.

### Cambio de método, no de hipótesis

El anteproyecto planificó contrastar H4 con valores SHAP de un modelo de árboles. El notebook 09 mostró que esos modelos tienen **R² negativo fuera de muestra**: atribuir importancia con un modelo que no predice es atribuir ruido. Se reemplaza el método, y la hipótesis se mantiene tal como fue formulada.

**Nuevo método.** El modelo de medición se ajusta con interacciones entre la ruralidad y dos bloques de variables: geográficas (distancia a la capital regional, densidad comunal y densidad de establecimientos en 5 km) y de dependencia (municipal, SLEP, particular pagado). En cada grupo se calcula la **contribución** de cada bloque: la desviación estándar, entre los establecimientos del grupo, de la parte de la pendiente (o de la caída) que aporta ese bloque. Es el análogo de la importancia SHAP en un modelo lineal.

**Corrección del piso de ruido.** La incertidumbre de los coeficientes infla la contribución, y más en el bloque con más coeficientes inciertos. Se resta a cada bloque su propio piso de ruido (sección 1 muestra por qué es necesario).

### Regla de decisión, fijada antes de ver los resultados

| Veredicto | Condición |
|---|---|
| Se sostiene | P(geografía > dependencia \| rural) ≥ 0,90 **y** P(dependencia > geografía \| urbano) ≥ 0,90 |
| Se refuta | ambas probabilidades ≤ 0,10 |
| No concluyente | cualquier otro caso |

**Potencia declarada de antemano.** La muestra tiene unos 152 establecimientos rurales (5 %). Un resultado no concluyente en la mitad rural es lo esperable, y no debe leerse como evidencia contra H4.

**Requiere** los notebooks 06 y 07 ejecutados. Tiempo: unos 5 minutos de simulaciones y 15 a 20 minutos de ajustes.

In [1]:
import sys, warnings
from pathlib import Path
for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]
candidatos = [Path.cwd() / "simce_panel" / "src", Path.cwd().parent / "src", Path.cwd() / "src"]
sys.path.insert(0, str(next(p for p in candidatos if (p / "simce_panel" / "__init__.py").exists())))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import simce_panel as sp
sp.configurar_logging()
cfg = sp.Config.desde_base(".")
B = sp.bayes

MATERIAS = ["mate", "lect"]
ITERACIONES = 1000
CADENAS = 2
UMBRAL = 0.90
BLOQUES_SIM = {"geograficas": ["geo1", "geo2", "geo3"], "dependencia": ["dep1", "dep2", "dep3"]}

## 1. Validación del método con datos simulados

Antes de aplicar el contraste a los datos reales se verifica que el método **detecta H4 cuando es verdadera** y **no la declara cuando es falsa**, con la misma proporción de rurales y la misma estructura de bloques que la tesis. Sin esta validación, un resultado no concluyente no permitiría distinguir entre falta de información y un método defectuoso.

In [2]:
def validar(h4_verdadera, semilla):
    L, X = B.simular_h4(1500, h4_verdadera=h4_verdadera, semilla=semilla)
    d = B.preparar_datos(L, col_n="n", covariables_escuela=X)
    m = B.ajustar(d, num_warmup=400, num_samples=400, num_chains=1)
    rur = X["rural"].reindex(np.asarray(d["rbd"])).to_numpy() == 1
    fila = {"escenario": "H4 verdadera" if h4_verdadera else "H4 falsa (nula)", "semilla": semilla}
    for corregir in (False, True):
        cr = B.contribucion_bloques(m, d, BLOQUES_SIM, rur, corregir_ruido=corregir)
        cu = B.contribucion_bloques(m, d, BLOQUES_SIM, ~rur, corregir_ruido=corregir)
        dec = B.decision_h4(cr, cu, umbral=UMBRAL)
        sufijo = "corregido" if corregir else "sin corregir"
        fila[f"P rural ({sufijo})"] = dec["P(geo > dep | rural)"]
        fila[f"veredicto ({sufijo})"] = dec["veredicto"]
    return fila

validacion = pd.DataFrame([validar(True, 0), validar(False, 1), validar(False, 2)])
validacion

23:41:47 | INFO    | simce_panel.bayes | Interacciones omitidas por falta de casos: ['dep2', 'dep3']
23:41:47 | INFO    | simce_panel.bayes | Covariables: 1500 de 1500 escuelas con datos completos.
23:41:48 | INFO    | simce_panel.bayes | Datos del modelo: 1500 escuelas, 12000 observaciones.
23:41:49 | INFO    | jax._src.xla_bridge | Unable to initialize backend 'tpu': UNIMPLEMENTED: LoadPjrtPlugin is not implemented on windows yet.
23:42:19 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 5.8, 'divergencias': 0, 'r_hat_max': 1.01, 'n_eff_min': 100}
23:42:19 | INFO    | simce_panel.bayes | Interacciones omitidas por falta de casos: ['dep2', 'dep3']
23:42:19 | INFO    | simce_panel.bayes | Covariables: 1500 de 1500 escuelas con datos completos.
23:42:19 | INFO    | simce_panel.bayes | Datos del modelo: 1500 escuelas, 12000 observaciones.
23:42:42 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 1.5, 'divergencias': 0, 'r_hat_max': 1.031, 'n_eff_min': 55}
23:42:42 | WARNING | sim

,escenario,semilla,P rural (sin corregir),veredicto (sin corregir),P rural (corregido),veredicto (corregido)
0,H4 verdadera,0,0.990,se sostiene,0.983,se sostiene
1,H4 falsa (nula),1,0.838,no concluyente,0.600,no concluyente
2,H4 falsa (nula),2,0.882,no concluyente,0.735,no concluyente


**Lectura.** Con H4 verdadera, el veredicto corregido debe ser "se sostiene". Con H4 falsa, debe ser "no concluyente" o "se refuta", y la probabilidad rural **sin corregir** debería quedar cerca del umbral: ese es el sesgo que motiva la corrección. Si la validación no se comporta así, no se interpreta la sección 3.

**Alcance de esta validación (auditoría del 24-oct-2026).** Son solo tres conjuntos simulados (uno con H4 verdadera y dos nulos), con una cadena de 400 iteraciones; uno de los ajustes nulos terminó con R-hat 1,031, sobre el umbral. La validación muestra que el método **puede** detectar H4 y que la corrección del piso de ruido funciona en esos casos, pero no estima tasas de error. Además, la simulación genera diferencias solo en la pendiente, no en la caída. Para Capstone II conviene repetirla con al menos 50 réplicas por escenario, dos cadenas y ambos resultados.

## 2. Datos y covariables

In [3]:
cov = sp.io_datos.leer_csv(cfg.ruta_salida("dataset_analitico_mate_covariables.csv")).set_index("rbd")
ancho = sp.io_datos.leer_csv(cfg.ruta_salida("simce_rbd_ancho_reducido_2015_2025.csv"))

GEO = ["log_dist_capital", "log_densidad_comunal", "log_dens5km"]
DEP = ["dep_municipal", "dep_slep", "dep_pagado"]
depe = cov["cod_depe1_cat"]
X = pd.DataFrame({
    "log_dist_capital": np.log1p(cov["dist_capital_regional_km"]),
    "log_densidad_comunal": cov["log_densidad_comunal"],
    "log_dens5km": np.log1p(cov["densidad_5km"]),
    "dep_municipal": depe.isin(["Municipal DAEM", "Municipal Corporación"]).astype(float),
    "dep_slep": depe.eq("Servicio Local de Educación").astype(float),
    "dep_pagado": depe.eq("Particular pagado").astype(float),
    "rural": cov["cod_rural_rbd_cat"].eq("Rural").astype(float),
}, index=cov.index)
X = B.agregar_interacciones(X, "rural", GEO + DEP)
print("Covariables del modelo:", list(X.columns))
print("Establecimientos rurales:", int(X["rural"].sum()))
print(pd.crosstab(cov["cod_rural_rbd_cat"], depe))

23:43:22 | INFO    | simce_panel.bayes | Interacciones omitidas por falta de casos: ['dep_pagado']


Covariables del modelo: ['log_dist_capital', 'log_densidad_comunal', 'log_dens5km', 'dep_municipal', 'dep_slep', 'dep_pagado', 'rural', 'log_dist_capital__x_rural', 'log_densidad_comunal__x_rural', 'log_dens5km__x_rural', 'dep_municipal__x_rural', 'dep_slep__x_rural']
Establecimientos rurales: 185
cod_depe1_cat      Corporación de administración delegada  \
cod_rural_rbd_cat                                           
Rural                                                   0   
Urbano                                                 70   

cod_depe1_cat      Municipal Corporación  Municipal DAEM  Particular pagado  \
cod_rural_rbd_cat                                                             
Rural                                  8              61                  5   
Urbano                               124             370                475   

cod_depe1_cat      Particular subvencionado  Servicio Local de Educación  
cod_rural_rbd_cat                                               

La tabla cruzada muestra cuántos establecimientos rurales hay en cada dependencia. Las interacciones con menos de diez casos (típicamente, particulares pagados rurales) se omiten porque no hay información para estimarlas.

**Nota (auditoría del 24-oct-2026).** El conteo "Establecimientos rurales: 185" y la tabla cruzada usan todo el archivo de covariables. En el modelo, después de exigir puntaje en 2022 y covariables completas, quedan unos 150 rurales (152 en el modelo del cuaderno 06). El informe debe citar la cifra del modelo.

## 3. Ajuste y contraste

In [4]:
def panel(materia):
    anios = [2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025]
    return pd.concat([pd.DataFrame({
        "rbd": ancho["rbd"], "cod_grupo_cat": ancho["cod_grupo_cat"], "anio": t,
        "y": pd.to_numeric(ancho[f"prom_{materia}2m_rbd_{t}"], errors="coerce"),
        "n": pd.to_numeric(ancho[f"nalu_{materia}2m_rbd_{t}"], errors="coerce")})
        for t in anios]).dropna(subset=["y"])

bloques = {"geograficas": GEO, "dependencia": DEP}
filas, detalle = [], {}
for materia in MATERIAS:
    d = B.preparar_datos(panel(materia), col_n="n", covariables_escuela=X)
    m = B.ajustar(d, tendencia_pre=True, controlar_pre=True, num_warmup=ITERACIONES,
                  num_samples=ITERACIONES, num_chains=CADENAS)
    print(materia, m.diagnosticos)
    rur = X["rural"].reindex(np.asarray(d["rbd"])).to_numpy() == 1
    for parametro, resultado in (("gam_b", "recuperación"), ("gam_d", "caída")):
        cr = B.contribucion_bloques(m, d, bloques, rur, parametro=parametro)
        cu = B.contribucion_bloques(m, d, bloques, ~rur, parametro=parametro)
        detalle[(materia, resultado)] = (cr, cu)
        filas.append({"materia": materia, "resultado": resultado,
                      **B.decision_h4(cr, cu, umbral=UMBRAL)})
tabla_h4 = pd.DataFrame(filas)
tabla_h4

23:43:22 | INFO    | simce_panel.bayes | Covariables: 2841 de 2866 escuelas con datos completos.
23:43:22 | INFO    | simce_panel.bayes | 5 escuelas excluidas por no tener puntaje en 2022.


23:43:22 | INFO    | simce_panel.bayes | Datos del modelo: 2836 escuelas, 22408 observaciones.
23:52:19 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 536.7, 'divergencias': 0, 'r_hat_max': 1.001, 'n_eff_min': 898}


mate {'segundos': 536.7, 'divergencias': 0, 'r_hat_max': 1.001, 'n_eff_min': 898}


23:52:19 | INFO    | simce_panel.bayes | Covariables: 2841 de 2866 escuelas con datos completos.
23:52:19 | INFO    | simce_panel.bayes | 6 escuelas excluidas por no tener puntaje en 2022.
23:52:20 | INFO    | simce_panel.bayes | Datos del modelo: 2835 escuelas, 22397 observaciones.
23:57:07 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 287.7, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 372}


lect {'segundos': 287.7, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 372}


,materia,resultado,P(geo > dep | rural),P(dep > geo | urbano),umbral,veredicto
0,mate,recuperación,0.267,0.634,0.9,no concluyente
1,mate,caída,0.490,0.982,0.9,no concluyente
2,lect,recuperación,0.598,0.361,0.9,no concluyente
3,lect,caída,0.184,0.670,0.9,no concluyente


**Criterio de convergencia:** cero divergencias y R-hat ≤ 1,01 en ambos ajustes antes de interpretar.

In [5]:
for (materia, resultado), (cr, cu) in detalle.items():
    print(f"=== {materia} · {resultado} (contribución en puntos{'/año' if resultado == 'recuperación' else ''})")
    print("  rural :", cr[["bloque", "de_contribucion", "ic"]].values.tolist())
    print("  urbano:", cu[["bloque", "de_contribucion", "ic"]].values.tolist())

=== mate · recuperación (contribución en puntos/año)
  rural : [['geograficas', 0.191, [0.0, 1.764]], ['dependencia', 0.789, [0.0, 2.243]]]
  urbano: [['geograficas', 0.0, [0.0, 0.353]], ['dependencia', 0.43, [0.0, 1.372]]]
=== mate · caída (contribución en puntos)
  rural : [['geograficas', 3.138, [0.0, 6.439]], ['dependencia', 3.072, [0.0, 6.322]]]
  urbano: [['geograficas', 0.225, [0.0, 1.011]], ['dependencia', 3.203, [0.824, 5.464]]]
=== lect · recuperación (contribución en puntos/año)
  rural : [['geograficas', 0.929, [0.0, 2.096]], ['dependencia', 0.539, [0.0, 1.775]]]
  urbano: [['geograficas', 0.29, [0.0, 0.509]], ['dependencia', 0.127, [0.0, 0.821]]]
=== lect · caída (contribución en puntos)
  rural : [['geograficas', 1.482, [0.0, 4.236]], ['dependencia', 3.2, [0.0, 5.817]]]
  urbano: [['geograficas', 0.257, [0.0, 0.827]], ['dependencia', 0.877, [0.0, 2.721]]]


## 4. Interpretación

**Cómo leer un resultado no concluyente.** La validación de la sección 1 muestra que el método detecta H4 cuando existe con la proporción de rurales de la tesis, siempre que las diferencias entre bloques sean marcadas. Un veredicto no concluyente indica que los datos no contienen diferencias de esa magnitud o que su incertidumbre es demasiado amplia, no que H4 sea falsa.

**Cota de lo que se puede excluir.** El extremo superior del intervalo de cada bloque en el grupo rural indica la mayor contribución compatible con los datos: contribuciones mayores quedan descartadas; menores, no.

Estos resultados son asociaciones dentro del modelo, condicionales al nivel previo y al grupo socioeconómico.

In [6]:
tabla_h4.to_csv(cfg.ruta_salida("h4_contraste.csv"), sep=";", index=False, encoding="utf-8-sig")
print("Guardado.")

Guardado.
